# ♟️ Chess-AI: Model Evaluation, Interactive Play & Benchmarks

This notebook provides the complete evaluation and testing suite:
- **Neural Network Position Evaluation & Transposition Caching**
- **Alpha-Beta Minimax Search with Policy Priors**
- **Interactive Play vs. AI** directly within Jupyter
- **Model Duels & PGN Match Generation**
- **Stockfish Gauntlet Benchmark** (Levels 1 to 8)


### 1. Imports and Device Setup

In [1]:
import os
import sys
import time
import uuid
import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import chess
import chess.pgn
from IPython.display import display, clear_output

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Evaluation Device: {device}")
if torch.cuda.is_available():
    print(f"[*] GPU: {torch.cuda.get_device_name(0)}")


### 2. Model Definition & Loading Weights

In [2]:
class ResBlock(nn.Module):
    def __init__(self, channels: int = 128):
        super(ResBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        out = F.relu(out)
        return out


class ChessResNet(nn.Module):
    def __init__(self, num_blocks: int = 10, hidden_channels: int = 128):
        super(ChessResNet, self).__init__()
        self.start_conv = nn.Conv2d(12, hidden_channels, kernel_size=3, padding=1)
        self.start_bn = nn.BatchNorm2d(hidden_channels)
        self.res_blocks = nn.ModuleList([ResBlock(hidden_channels) for _ in range(num_blocks)])
        
        # Policy Head (76 planes x 64 squares = 4,864)
        self.policy_conv = nn.Conv2d(hidden_channels, 76, kernel_size=1)
        self.policy_bn = nn.BatchNorm2d(76)
        self.policy_fc = nn.Linear(76 * 8 * 8, 4864)
        
        # Value Head
        self.eval_conv = nn.Conv2d(hidden_channels, 1, kernel_size=1)
        self.eval_bn = nn.BatchNorm2d(1)
        self.eval_fc1 = nn.Linear(8 * 8, 64)
        self.eval_fc2 = nn.Linear(64, 1)

    def forward(self, x: torch.Tensor):
        x = F.relu(self.start_bn(self.start_conv(x)))
        for block in self.res_blocks:
            x = block(x)
            
        p = F.relu(self.policy_bn(self.policy_conv(x)))
        p = p.view(-1, 4864)
        policy_out = self.policy_fc(p)
        
        v = F.relu(self.eval_bn(self.eval_conv(x)))
        v = v.view(-1, 64)
        v = F.relu(self.eval_fc1(v))
        v = self.eval_fc2(v)
        value_out = torch.tanh(v)
        return policy_out, value_out


# Initialize model
model = ChessResNet(num_blocks=10, hidden_channels=128).to(device)

# Try loading weights from common paths
checkpoint_paths = ['models/chess_model_v3.pth', 'models/chess_model.pth', 'chess_model.pth', 'Models/chess_model_v3.pth']
loaded = False
for cp in checkpoint_paths:
    if os.path.exists(cp):
        print(f"[*] Loading weights from: {cp}")
        model.load_state_dict(torch.load(cp, map_location=device, weights_only=True))
        loaded = True
        break

if not loaded:
    print("[*] Warning: No weights file found. Running with randomly initialized weights.")

model.eval()


### 3. Board Encoding, Transposition Caching & Alpha-Beta Minimax Search

In [3]:
# Global transposition and search caches
evaluation_cache = {}
search_cache = {}
eval_cache_hits = 0
eval_cache_misses = 0
search_cache_hits = 0
search_cache_misses = 0


def board_to_tensor(board: chess.Board) -> np.ndarray:
    tensor = np.zeros((12, 64), dtype=np.float32)
    for sq, piece in board.piece_map().items():
        layer = (piece.piece_type - 1) + (0 if piece.color == chess.WHITE else 6)
        tensor[layer][sq] = 1.0
    tensor = tensor.reshape(12, 8, 8)
    return np.flip(tensor, axis=1).copy()


def move_to_index(move: chess.Move) -> int:
    from_sq, to_sq = move.from_square, move.to_square
    r0, c0 = divmod(from_sq, 8)
    r1, c1 = divmod(to_sq, 8)
    dr, dc = r1 - r0, c1 - c0

    knight_moves = [(2, 1), (1, 2), (-1, 2), (-2, 1), (-2, -1), (-1, -2), (1, -2), (2, -1)]
    if (dr, dc) in knight_moves:
        plane = 56 + knight_moves.index((dr, dc))
        return plane * 64 + from_sq

    if move.promotion is not None:
        dir_idx = 0 if dc == 0 else (1 if dc == -1 else 2)
        promo_map = {chess.KNIGHT: 0, chess.BISHOP: 1, chess.ROOK: 2, chess.QUEEN: 3}
        piece_idx = promo_map[move.promotion]
        plane = 64 + dir_idx * 4 + piece_idx
        return plane * 64 + from_sq

    directions = [(1, 0), (1, 1), (0, 1), (-1, 1), (-1, 0), (-1, -1), (0, -1), (1, -1)]
    for dir_idx, (drd, dcd) in enumerate(directions):
        for dist in range(1, 8):
            if dr == drd * dist and dc == dcd * dist:
                plane = dir_idx * 7 + (dist - 1)
                return plane * 64 + from_sq
    return None


def get_model_evaluation(model: nn.Module, board: chess.Board, device: torch.device):
    global eval_cache_hits, eval_cache_misses
    cache_key = (id(model), board._transposition_key())
    if cache_key in evaluation_cache:
        eval_cache_hits += 1
        return evaluation_cache[cache_key]
    eval_cache_misses += 1

    is_black = (board.turn == chess.BLACK)
    eval_board = board.mirror() if is_black else board
    np_board = board_to_tensor(eval_board)
    input_tensor = torch.from_numpy(np_board).unsqueeze(0).to(device, non_blocking=True)

    with torch.inference_mode():
        policy_out, value_out = model(input_tensor)

    final_eval = value_out.item() * (-1 if is_black else 1)
    result = (policy_out, final_eval)
    evaluation_cache[cache_key] = result
    if len(evaluation_cache) > 1_000_000:
        evaluation_cache.clear()
    return result


def minimax(board: chess.Board, depth: int, alpha: float, beta: float, is_maximizing: bool, model: nn.Module, device: torch.device):
    global search_cache_hits, search_cache_misses
    cache_key = (board._transposition_key(), depth, is_maximizing)
    if cache_key in search_cache:
        search_cache_hits += 1
        return search_cache[cache_key]
    search_cache_misses += 1

    if board.is_checkmate():
        score = (-10000.0 - depth) if is_maximizing else (10000.0 + depth)
        result = (score, [])
        search_cache[cache_key] = result
        return result

    if board.is_stalemate() or board.is_insufficient_material() or board.can_claim_draw() or board.is_repetition(3):
        result = (0.0, [])
        search_cache[cache_key] = result
        return result

    policy_logits, value_score = get_model_evaluation(model, board, device)
    if depth <= 0:
        result = (value_score, [])
        search_cache[cache_key] = result
        return result

    legal_moves = list(board.legal_moves)
    is_black = (board.turn == chess.BLACK)
    policy_scores = policy_logits.squeeze(0)

    moves_with_scores = []
    for move in legal_moves:
        m = chess.Move(chess.square_mirror(move.from_square), chess.square_mirror(move.to_square), move.promotion) if is_black else move
        idx = move_to_index(m)
        if idx is not None:
            moves_with_scores.append((policy_scores[idx].item(), move))

    moves_with_scores.sort(key=lambda x: x[0], reverse=True)
    moves_with_scores = moves_with_scores[:8]
    best_path = []

    if is_maximizing:
        best_eval = float("-inf")
        for _, move in moves_with_scores:
            board.push(move)
            eval_score, path = minimax(board, depth - 1, alpha, beta, False, model, device)
            board.pop()
            if eval_score > best_eval:
                best_eval = eval_score
                best_path = [move] + path
            alpha = max(alpha, best_eval)
            if beta <= alpha:
                break
        result = (best_eval, best_path)
        search_cache[cache_key] = result
        return result
    else:
        best_eval = float("inf")
        for _, move in moves_with_scores:
            board.push(move)
            eval_score, path = minimax(board, depth - 1, alpha, beta, True, model, device)
            board.pop()
            if eval_score < best_eval:
                best_eval = eval_score
                best_path = [move] + path
            beta = min(beta, best_eval)
            if beta <= alpha:
                break
        result = (best_eval, best_path)
        search_cache[cache_key] = result
        return result


def get_best_move(board_state: chess.Board, lookahead_depth: int, model: nn.Module, device: torch.device, candidate_pruning: int = 12):
    global search_cache, search_cache_hits, search_cache_misses
    search_cache.clear()
    search_cache_hits = 0
    search_cache_misses = 0

    is_white_turn = (board_state.turn == chess.WHITE)
    best_move = None
    policy_out, _ = get_model_evaluation(model, board_state, device)
    policy_scores = policy_out.squeeze(0)
    legal_moves = list(board_state.legal_moves)
    if not legal_moves:
        return None

    def move_priority(move: chess.Move) -> float:
        m = chess.Move(chess.square_mirror(move.from_square), chess.square_mirror(move.to_square), move.promotion) if not is_white_turn else move
        idx = move_to_index(m)
        return float("-inf") if idx is None else policy_scores[idx].item()

    legal_moves.sort(key=move_priority, reverse=True)
    legal_moves = legal_moves[:candidate_pruning]

    alpha, beta = float("-inf"), float("inf")
    best_val = float("-inf") if is_white_turn else float("inf")

    for move in legal_moves:
        board_state.push(move)
        val, _ = minimax(board_state, lookahead_depth - 1, alpha, beta, not is_white_turn, model, device)
        board_state.pop()

        if is_white_turn:
            if val > best_val:
                best_val, best_move = val, move
            alpha = max(alpha, best_val)
        else:
            if val < best_val:
                best_val, best_move = val, move
            beta = min(beta, best_val)

    return best_move if best_move is not None else (legal_moves[0] if legal_moves else None)


### 4. Interactive Human vs. Machine Game

Run this cell to play against your trained AI model directly inside the notebook with rendered graphical chess boards!

In [4]:
def play_vs_human_in_notebook(model, player_color=chess.BLACK, depth=3):
    board = chess.Board()
    player_is_black = (player_color == chess.BLACK)
    
    print("=== CHESS AI: MAN VS MACHINE ===")
    print(f"You are playing as {'BLACK' if player_is_black else 'WHITE'}.")
    print("Enter moves in SAN (e.g. e4, Nf6, O-O) or UCI (e2e4).")
    print("Type 'quit' to exit.\n")
    
    while not board.is_game_over():
        clear_output(wait=True)
        display(board if not player_is_black else board.mirror())
        
        is_ai_turn = (board.turn != player_color)
        
        if is_ai_turn:
            print("[*] AI is calculating move...")
            with torch.no_grad():
                ai_move = get_best_move(board, lookahead_depth=depth, model=model, device=device)
            if ai_move and ai_move in board.legal_moves:
                print(f"[*] AI played: {board.san(ai_move)}")
                board.push(ai_move)
            else:
                print("[!] AI resigned!")
                break
        else:
            user_input = input(f"Your move ({'Black' if player_is_black else 'White'}): ").strip()
            if user_input.lower() in ['quit', 'exit', 'resign']:
                print("Game terminated by player.")
                break
            try:
                move = board.parse_san(user_input)
                board.push(move)
            except ValueError:
                try:
                    move = chess.Move.from_uci(user_input.lower())
                    if move in board.legal_moves:
                        board.push(move)
                    else:
                        print("Illegal move. Try again.")
                        time.sleep(1.5)
                except ValueError:
                    print("Invalid notation. Try SAN or UCI.")
                    time.sleep(1.5)
                    
    clear_output(wait=True)
    display(board)
    print("=== GAME OVER ===")
    print(f"Result: {board.result()}")

# To start playing, uncomment below:
# play_vs_human_in_notebook(model, player_color=chess.BLACK, depth=3)


### 5. Model Duels & PGN Match Generation

Pit two models against each other (or test two different checkpoints/depths) and automatically export full PGN game logs to `pgn_exports/`.

In [5]:
def run_model_duel(model_a, model_b, num_games=2, depth=3, save_dir="pgn_exports"):
    results = {"Model_A": 0, "Model_B": 0, "Draws": 0}
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)
    all_games = []

    for game_num in range(num_games):
        board = chess.Board()
        a_is_white = (game_num % 2 == 0)

        game = chess.pgn.Game()
        game.headers["Event"] = "Model Duel"
        game.headers["Site"] = "Local"
        game.headers["Date"] = datetime.datetime.now().strftime("%Y.%m.%d")
        game.headers["Round"] = str(game_num + 1)
        game.headers["White"] = "Model_A" if a_is_white else "Model_B"
        game.headers["Black"] = "Model_B" if a_is_white else "Model_A"
        node = game

        while not board.is_game_over() and not board.can_claim_draw():
            is_ai_a_turn = ((board.turn == chess.WHITE) == a_is_white)
            current_model = model_a if is_ai_a_turn else model_b
            
            with torch.no_grad():
                move = get_best_move(board, lookahead_depth=depth, model=current_model, device=device)

            if move and move in board.legal_moves:
                board.push(move)
                node = node.add_variation(move)
            else:
                break

        outcome = board.outcome()
        if outcome is None or outcome.winner is None:
            results["Draws"] += 1
            result_str = "1/2-1/2"
        elif (outcome.winner == chess.WHITE and a_is_white) or (outcome.winner == chess.BLACK and not a_is_white):
            results["Model_A"] += 1
            result_str = "1-0" if a_is_white else "0-1"
        else:
            results["Model_B"] += 1
            result_str = "0-1" if a_is_white else "1-0"

        game.headers["Result"] = result_str
        all_games.append(game)

    combined_name = f"duel_match_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:6]}.pgn"
    combined_file = save_path / combined_name
    with open(combined_file, "w", encoding="utf-8") as f:
        for g in all_games:
            exporter = chess.pgn.FileExporter(f)
            g.accept(exporter)
            f.write("\n\n")

    print(f"[*] Duel Complete: Model_A {results['Model_A']} - {results['Model_B']} Model_B ({results['Draws']} Draws)")
    print(f"[*] Match PGN saved to: {combined_file}")
    return results

# Example duel execution:
# run_model_duel(model, model, num_games=2, depth=3)


### 6. Stockfish Benchmark Gauntlet (Levels 1 to 8)

Tests the model against Stockfish configured across different skill levels.

In [6]:
import chess.engine

STOCKFISH_PATH = os.environ.get("STOCKFISH_PATH", "stockfish/stockfish-windows-x86-64-avx2.exe")

SKILL_LEVELS = {
    "Level 1": 0,
    "Level 2": 3,
    "Level 3": 6,
    "Level 4": 9,
    "Level 5": 12,
    "Level 6": 15,
    "Level 7": 18,
    "Level 8": 20
}

def run_stockfish_gauntlet(stockfish_path: str, model: nn.Module, games_per_level: int = 2):
    if not os.path.exists(stockfish_path):
        print(f"[!] Stockfish binary not found at: {stockfish_path}")
        print("[!] Set the STOCKFISH_PATH environment variable or place the binary in 'stockfish/' to run the gauntlet.")
        return {}
        
    overall_results = {}
    with chess.engine.SimpleEngine.popen_uci(stockfish_path) as engine:
        for level_name, skill in SKILL_LEVELS.items():
            engine.configure({"Skill Level": skill})
            wins, losses, draws = 0, 0, 0
            
            for game_idx in range(games_per_level):
                board = chess.Board()
                while not board.is_game_over():
                    if board.turn == chess.WHITE:
                        move = get_best_move(board, lookahead_depth=3, model=model, device=device)
                    else:
                        res = engine.play(board, chess.engine.Limit(time=0.1))
                        move = res.move
                    if move and move in board.legal_moves:
                        board.push(move)
                    else:
                        break
                        
                outcome = board.outcome()
                if outcome and outcome.winner == chess.WHITE:
                    wins += 1
                elif outcome and outcome.winner == chess.BLACK:
                    losses += 1
                else:
                    draws += 1
                    
            overall_results[level_name] = {"Wins": wins, "Losses": losses, "Draws": draws}
            print(f"{level_name}: +{wins} -{losses} ={draws}")
            
    return overall_results

# To run the gauntlet, uncomment below:
# run_stockfish_gauntlet(STOCKFISH_PATH, model, games_per_level=2)
